# 00 — Preprocessing

Converts raw NASA Randomized Battery Usage `.mat` files into a single HDF5 dataset used by all model notebooks.

**Run this notebook once** before running any model notebook. Output files:
- `data/processed/sequences.h5` — 450K × (301, 3) float32 sequences + SOH labels (~1.52 GB uncompressed)
- `data/processed/norm_stats.json` — per-channel mean/std computed on training split only

**Raw data location:** `../../11. Randomized Battery Usage Data Set/` (relative to this notebook)

In [ ]:
import os
import sys
import json
import h5py
import numpy as np

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.preprocessing import build_sequence_dataset
from src.dataset import BatterySOHDataset, DEFAULT_TEST_BATTERIES

DATA_DIR   = '../../11. Randomized Battery Usage Data Set'
H5_PATH    = '../data/processed/sequences.h5'
STATS_PATH = '../data/processed/norm_stats.json'

print(f'Raw data dir : {os.path.abspath(DATA_DIR)}')
print(f'Raw data exists: {os.path.isdir(DATA_DIR)}')
print(f'Output HDF5  : {os.path.abspath(H5_PATH)}')

## Step 1 — Build HDF5 Dataset

Iterates all 28 `.mat` files, extracts charge/discharge random-walk steps, resamples each to 301 time points, and labels each step with the SOH from the next reference discharge.

Expected output: ~450K steps, ~2–3 minutes on first run.

In [ ]:
if os.path.exists(H5_PATH):
    print(f'HDF5 already exists at {H5_PATH} — skipping build.')
    print('Delete the file and re-run this cell to rebuild from scratch.')
else:
    build_sequence_dataset(
        data_dir=DATA_DIR,
        output_path=H5_PATH,
    )

## Step 2 — Verify HDF5 Contents

In [ ]:
with h5py.File(H5_PATH, 'r') as f:
    n   = f['X'].shape[0]
    print('Datasets:')
    for key in f.keys():
        print(f'  {key:12s} {str(f[key].shape):25s} {f[key].dtype}')
    print()
    print('Attributes:')
    for k, v in f.attrs.items():
        print(f'  {k}: {v}')
    print()
    size_gb = f['X'].shape[0] * 301 * 3 * 4 / 1e9
    print(f'Total samples : {n:,}')
    print(f'X uncompressed: {size_gb:.2f} GB')
    print(f'SOH range     : {f["y"][:].min():.1f}% – {f["y"][:].max():.1f}%')

## Step 3 — Battery-Level Split

Prints the train/val/test split counts. Test batteries are held out entirely to prevent data leakage.

In [ ]:
train_idx, val_idx, test_idx = BatterySOHDataset.create_splits_3way(H5_PATH)

print(f'Train samples : {len(train_idx):>7,}')
print(f'Val samples   : {len(val_idx):>7,}')
print(f'Test samples  : {len(test_idx):>7,}')
print(f'Total         : {len(train_idx)+len(val_idx)+len(test_idx):>7,}')
print()
print(f'Test batteries (held out): {DEFAULT_TEST_BATTERIES}')

## Step 4 — Compute Normalization Stats

Per-channel mean and std computed **on training split only**. Test split is normalized using these same stats to prevent leakage.

In [ ]:
if os.path.exists(STATS_PATH):
    print(f'norm_stats.json already exists — skipping computation.')
else:
    tmp_ds = BatterySOHDataset(H5_PATH, indices=train_idx, normalize=False)
    tmp_ds.compute_normalization_stats(STATS_PATH)
    print(f'Saved to {STATS_PATH}')

with open(STATS_PATH) as f:
    stats = json.load(f)

channels = ['voltage', 'current', 'temperature']
print(f'\n{"Channel":<12} {"Mean":>10} {"Std":>10}')
print('-' * 34)
for i, ch in enumerate(channels):
    print(f'{ch:<12} {stats["mean"][i]:>10.4f} {stats["std"][i]:>10.4f}')

## Step 5 — Sanity Check: Sample Waveforms

Plots one charge and one discharge step to verify the resampling looks correct.

In [ ]:
import matplotlib.pyplot as plt

with h5py.File(H5_PATH, 'r') as f:
    X_sample = f['X'][:500]          # first 500 steps
    y_sample = f['y'][:500]
    bid_sample = f['battery_id'][:500]

fig, axes = plt.subplots(2, 3, figsize=(13, 6))
channel_names = ['Voltage (V)', 'Current (A)', 'Temperature (°C)']

for row, idx in enumerate([0, 1]):
    for col, ch_name in enumerate(channel_names):
        ax = axes[row, col]
        ax.plot(X_sample[idx, :, col], lw=1.2)
        ax.set_title(f'Step {idx} — {ch_name}\nBattery: {bid_sample[idx].decode()}  SOH: {y_sample[idx]:.1f}%')
        ax.set_xlabel('Time point (0–300)')
        ax.grid(alpha=0.3)

fig.suptitle('Sample Waveforms (raw, unnormalized)', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

## Step 6 — SOH Label Distribution

In [ ]:
with h5py.File(H5_PATH, 'r') as f:
    y_all = f['y'][:]

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(y_all, bins=80, color='steelblue', edgecolor='white', alpha=0.85)
ax.set_xlabel('SOH (%)')
ax.set_ylabel('Step count')
ax.set_title(f'SOH Label Distribution  (N={len(y_all):,})')
ax.axvline(y_all.mean(), color='red', lw=1.5, ls='--', label=f'Mean={y_all.mean():.1f}%')
ax.legend()
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.show()

print(f'Mean SOH : {y_all.mean():.2f}%')
print(f'Std SOH  : {y_all.std():.2f}%')
print(f'Min SOH  : {y_all.min():.2f}%')
print(f'Max SOH  : {y_all.max():.2f}%')

## Summary

Preprocessing is complete. The files below are ready for all model notebooks:

| File | Description |
|------|-------------|
| `data/processed/sequences.h5` | 450K steps × (301, 3) float32 + SOH labels |
| `data/processed/norm_stats.json` | Per-channel mean/std (training split only) |